# Skill Library Visualization / 技能库可视化

Inspects a `BoundedSkillLibrary` after training:
- Distribution of skills across GPU / CPU / SSD tiers
- Usage counts and average rewards
- t-SNE / PCA of LoRA flattened weights

在 Stage 4+ 训练后打开这份 notebook，观察技能库的分布、使用统计和聚类。

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.memory import BoundedSkillLibrary, SkillLibraryBudget, SkillEntry, SkillWeights

In [ ]:
# ---- Load or synthesize a skill library ----
# In real Stage 4+ workflow, you'd load a checkpoint. Here we synthesize.
budget = SkillLibraryBudget(gpu_capacity=8, cpu_capacity=16, ssd_max_shards=2, ssd_shard_size=8, merge_similarity_threshold=0.99)
lib = BoundedSkillLibrary(budget=budget, skill_shape=(16, 4, 16), device='cpu', archive_dir='./tmp_skills')
torch.manual_seed(0)
for i in range(24):
    s = lib.new_skill(tag=f'demo-{i}')
    for _ in range(np.random.randint(0, 20)):
        s.record_use(reward=float(np.random.rand()))
    lib.add(s)

print(lib.stats())

In [ ]:
# ---- Usage vs avg-reward scatter ----
gpu_entries = lib.top_k()
usage = [e.usage_count for e in gpu_entries]
avg_r = [e.avg_reward for e in gpu_entries]
tags = [e.tag for e in gpu_entries]

fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(usage, avg_r, s=80, alpha=0.7)
for i, tag in enumerate(tags):
    ax.annotate(tag, (usage[i], avg_r[i]), fontsize=8)
ax.set_xlabel('usage count')
ax.set_ylabel('avg reward')
ax.set_title('GPU-tier skills · usage vs reward')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Flattened LoRA weights heatmap (top skills) ----
if gpu_entries:
    flats = np.stack([
        np.concatenate([e.weights.A.flatten().numpy(), e.weights.B.flatten().numpy()])
        for e in gpu_entries
    ])
    plt.figure(figsize=(10, 4))
    plt.imshow(flats, aspect='auto', cmap='RdBu_r')
    plt.colorbar(label='weight')
    plt.ylabel('skill index (GPU tier)')
    plt.xlabel('flat parameter index')
    plt.title('GPU-tier skill weights (LoRA A ‖ B concatenated)')
    plt.tight_layout()
    plt.show()

In [ ]:
# ---- Pairwise cosine similarity (any near-duplicates hiding?) ----
import torch.nn.functional as F
if len(gpu_entries) >= 2:
    mat = torch.tensor(flats)
    sim = F.cosine_similarity(mat.unsqueeze(1), mat.unsqueeze(0), dim=-1).numpy()
    plt.figure(figsize=(6, 5))
    plt.imshow(sim, vmin=-1, vmax=1, cmap='RdBu_r')
    plt.colorbar(label='cosine similarity')
    plt.title('GPU-tier skill similarity matrix')
    plt.tight_layout()
    plt.show()

    print(f'max off-diagonal similarity: {(sim - np.eye(len(gpu_entries))).max():.3f}')
    print(f'merge threshold: {budget.merge_similarity_threshold}')